# 🚀 PhoBERT-v2 SOTA Inference Pipeline: Sentiment Analysis on Vietnamese Student Feedback

Notebook này hướng dẫn sử dụng mô hình **PhoBERT-v2 SOTA** đã được huấn luyện để thực hiện dự đoán (Inference / Serving):
- Tự động quét và giải nén weights từ thư mục `models/phobert_model_weights.zip` hoặc `./artifacts/phobert-sota`.
- Áp dụng trọn vẹn pipeline tiền xử lý `clean_text_vietnamese()` và **Topic-Prompt Injection** (`Chủ đề: [Topic] | [Sentence]`).
- Hỗ trợ 2 chế độ dự đoán:
  1. **Standard Argmax** (Precision cho NEUTRAL đạt **75.44%**, Accuracy đạt **94.25%**).
  2. **Threshold Tuning ($\tau = 0.08$)** (Recall cho NEUTRAL đạt **70.06%**, Macro F1 đạt **85.23%**).
- Hỗ trợ dự đoán câu đơn (Single Sentence) và theo lô lớn (Batch Inference với Pandas DataFrame).
- Đo lường độ trễ (Latency Benchmark) phục vụ triển khai API MLOps.


In [ ]:
# Cell 1: Cài đặt Dependencies & Tự động quét / giải nén Model Weights từ models/
import os
import zipfile
import torch

# 1. Cài đặt thư viện cần thiết
!pip install --upgrade pip -q
!pip install --no-cache-dir torch transformers pandas -q

# 2. Kiểm tra thiết bị phần cứng (GPU CUDA / Apple Silicon MPS / CPU)
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'⚡ Đang sử dụng GPU: {torch.cuda.get_device_name(0)}')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    print('⚡ Đang sử dụng Apple Silicon GPU (MPS)')
else:
    device = torch.device('cpu')
    print('💻 Đang sử dụng CPU')

# 3. Tự động tìm và giải nén weights từ models/phobert_model_weights.zip
MODEL_DIR = './artifacts/phobert-sota'
possible_zip_paths = [
    'models/phobert_model_weights.zip',
    '../models/phobert_model_weights.zip',
    './phobert_model_weights.zip',
    '/kaggle/working/AI_in_DevOps-DataOps-MLOps_Final_Project/models/phobert_model_weights.zip',
    '/kaggle/working/phobert_model_weights.zip'
]

model_safetensors_path = os.path.join(MODEL_DIR, 'model.safetensors')
pytorch_model_path = os.path.join(MODEL_DIR, 'pytorch_model.bin')

if not os.path.exists(model_safetensors_path) and not os.path.exists(pytorch_model_path):
    found_zip = None
    for zp in possible_zip_paths:
        if os.path.exists(zp):
            found_zip = zp
            break

    if found_zip:
        print(f'📦 Tìm thấy file nén: {found_zip}. Đang giải nén...')
        with zipfile.ZipFile(found_zip, 'r') as zip_ref:
            zip_ref.extractall('.')
        print(f'✅ Đã giải nén thành công vào: {MODEL_DIR}')
    else:
        print('⚠️ Chưa tìm thấy file weights local.')
        print('👉 Bạn có thể tải weights trực tiếp từ: https://drive.google.com/drive/folders/1hkdcZTRQKmz2BXsgPseg1Y85ocgvbqz9?usp=sharing')
else:
    print(f'✅ Đã có sẵn model weights tại: {MODEL_DIR}')


⚡ Đang sử dụng Apple Silicon GPU (MPS)
📦 Tìm thấy file nén: ../models/phobert_model_weights.zip. Đang giải nén...
✅ Đã giải nén thành công vào: ./artifacts/phobert-sota


In [5]:
# Cell 2: Pipeline Tiền Xử Lý Dữ Liệu & Topic-Prompt Context Injection
import re
from typing import List, Dict, Union

def clean_text_vietnamese(text: str) -> str:
    """
    Chuẩn hóa dữ liệu văn bản nhận xét tiếng Việt:
    1. doubledot -> ':' (ví dụ: '11doubledot55' -> '11:55', 'doubledot' -> ':')
    2. fraction  -> '/' (ví dụ: 'thầy fraction cô' -> 'thầy/cô')
    3. wzjwz<id> -> '[ANON]' (ví dụ: 'thầy wzjwz208' -> 'thầy [ANON]')
    """
    if not isinstance(text, str):
        return ''
    # 1. Thay 'doubledot' -> ':'
    text = re.sub(r'doubledot', ':', text, flags=re.IGNORECASE)
    # 2. Thay 'fraction' -> '/'
    text = re.sub(r'\bfraction\b', '/', text, flags=re.IGNORECASE)
    # 3. Thay mã ẩn danh 'wzjwz<id>' -> '[ANON]'
    text = re.sub(r'wzjwz\d+', '[ANON]', text, flags=re.IGNORECASE)
    return text.strip()

def format_input_with_topic(sentence: str, topic: str = 'others') -> str:
    """
    Đóng gói câu cùng thông tin Topic theo format đã huấn luyện trên PhoBERT-v2:
    Format: 'Chủ đề: [Topic] | [Cleaned Sentence]'
    """
    cleaned = clean_text_vietnamese(sentence)
    topic_clean = topic.strip() if topic else 'others'
    return f'Chủ đề: {topic_clean} | {cleaned}'

# Kiểm thử nhanh hàm tiền xử lý
sample_raw = 'thầy wzjwz208 dạy từ 7doubledot30 đến 11doubledot55 thầy fraction cô rất nhiệt tình .'
print('📝 Câu gốc:       ', sample_raw)
print('✨ Câu chuẩn hóa: ', clean_text_vietnamese(sample_raw))
print('🎯 Input PhoBERT: ', format_input_with_topic(sample_raw, topic='lecturer'))


📝 Câu gốc:        thầy wzjwz208 dạy từ 7doubledot30 đến 11doubledot55 thầy fraction cô rất nhiệt tình .
✨ Câu chuẩn hóa:  thầy [ANON] dạy từ 7:30 đến 11:55 thầy / cô rất nhiệt tình .
🎯 Input PhoBERT:  Chủ đề: lecturer | thầy [ANON] dạy từ 7:30 đến 11:55 thầy / cô rất nhiệt tình .


In [ ]:
# Cell 3: Xây dựng PhoBERTPredictor Class phục vụ Inference
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

LABEL_MAP = {0: 'NEGATIVE', 1: 'NEUTRAL', 2: 'POSITIVE'}

class PhoBERTPredictor:
    def __init__(self, model_dir: str = './artifacts/phobert-sota', base_model: str = 'vinai/phobert-base-v2', max_length: int = 128, device: torch.device = device):
        self.device = device
        self.max_length = max_length
        print(f'⏳ Loading Tokenizer & Model from: {model_dir}...')

        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
            self.model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        except Exception as e:
            print(f'⚠️ Chưa load được local ({e}), fallback sang tokenizer {base_model}...')
            self.tokenizer = AutoTokenizer.from_pretrained(base_model)
            self.model = AutoModelForSequenceClassification.from_pretrained(model_dir if os.path.exists(model_dir) else base_model, num_labels=3)

        self.model.to(self.device)
        self.model.eval()
        print('✅ PhoBERTPredictor đã sẵn sàng phục vụ!')

    @torch.no_grad()
    def predict_one(self, sentence: str, topic: str = 'others', threshold_mode: bool = True, tau_neutral: float = 0.08) -> Dict:
        """
        Dự đoán cảm xúc cho 1 câu đơn lẻ.
        """
        formatted_text = format_input_with_topic(sentence, topic)
        inputs = self.tokenizer(
            formatted_text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        ).to(self.device)

        outputs = self.model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1).squeeze(0).cpu().numpy()

        # 1. Quyết định nhãn theo Threshold Tuning hoặc Standard Argmax
        if threshold_mode:
            if probs[1] >= tau_neutral:  # NEUTRAL
                pred_id = 1
            else:
                pred_id = 0 if probs[0] >= probs[2] else 2
        else:
            pred_id = int(probs.argmax())

        return {
            'raw_sentence': sentence,
            'topic': topic,
            'formatted_input': formatted_text,
            'predicted_label': LABEL_MAP[pred_id],
            'predicted_id': pred_id,
            'confidence': float(probs[pred_id]),
            'probabilities': {
                'NEGATIVE': float(probs[0]),
                'NEUTRAL': float(probs[1]),
                'POSITIVE': float(probs[2])
            }
        }

    @torch.no_grad()
    def predict_batch(self, sentences: List[str], topics: List[str] = None, batch_size: int = 32, threshold_mode: bool = True, tau_neutral: float = 0.08) -> List[Dict]:
        """
        Dự đoán cảm xúc theo batch lớn với tốc độ tối ưu.
        """
        if topics is None:
            topics = ['others'] * len(sentences)

        formatted_texts = [format_input_with_topic(s, t) for s, t in zip(sentences, topics)]
        results = []

        for i in range(0, len(formatted_texts), batch_size):
            batch_texts = formatted_texts[i : i + batch_size]
            batch_sentences = sentences[i : i + batch_size]
            batch_topics = topics[i : i + batch_size]

            inputs = self.tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors='pt'
            ).to(self.device)

            outputs = self.model(**inputs)
            probs_batch = F.softmax(outputs.logits, dim=-1).cpu().numpy()

            for s, t, fmt, probs in zip(batch_sentences, batch_topics, batch_texts, probs_batch):
                if threshold_mode:
                    pred_id = 1 if probs[1] >= tau_neutral else (0 if probs[0] >= probs[2] else 2)
                else:
                    pred_id = int(probs.argmax())

                results.append({
                    'raw_sentence': s,
                    'topic': t,
                    'predicted_label': LABEL_MAP[pred_id],
                    'confidence': float(probs[pred_id]),
                    'prob_negative': float(probs[0]),
                    'prob_neutral': float(probs[1]),
                    'prob_positive': float(probs[2])
                })
        return results

# Khởi tạo predictor
predictor = PhoBERTPredictor(model_dir=MODEL_DIR, device=device)


⏳ Loading Tokenizer & Model from: ./artifacts/phobert-sota...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 13248.08it/s]


✅ PhoBERTPredictor đã sẵn sàng phục vụ!


In [ ]:
# Cell 4: Thử Nghiệm Dự Đoán Trên Các Mẫu Câu Thực Tế (Test Cases)

test_cases = [
    # 1. Khen ngợi rõ ràng (POSITIVE)
    ('thầy dạy rất có tâm, giảng bài dễ hiểu và nhiệt tình hỗ trợ sinh viên', 'lecturer'),
    ('môn học cực kỳ bổ ích, em học được rất nhiều kiến thức thực tế', 'program'),

    # 2. Phàn nàn gay gắt (NEGATIVE)
    ('máy lạnh phòng wzjwz208 bị hỏng suốt cả học kỳ mà không ai sửa doubledot', 'facility'),
    ('slide bài giảng quá sơ sài, thầy giảng buồn ngủ và khó hiểu fraction không có thực hành', 'lecturer'),

    # 3. Câu trung tính / Nước đôi / Câu ngắn (NEUTRAL - Điểm then chốt)
    ('bài tập trên lớp .', 'others'),
    ('nghiên cứu khoa học .', 'others'),
    ('thầy dạy bình thường, kiến thức vừa phải tạm chấp nhận được', 'lecturer'),
    ('chia sẻ .', 'others'),

    # 4. Câu có rác artifact từ bộ dữ liệu gốc
    ('thời gian học từ 11doubledot55 đến 13doubledot30 hơi bất tiện', 'program')
]

print('🔍 --- KẾT QUẢ DỰ ĐOÁN TỪNG CÂU VỚI PHOBERT-V2 (Threshold tau=0.08) --- \n')

for text, topic in test_cases:
    res = predictor.predict_one(text, topic=topic, threshold_mode=True, tau_neutral=0.08)
    label = res['predicted_label']
    conf = res['confidence'] * 100
    p_neg = res['probabilities']['NEGATIVE'] * 100
    p_neu = res['probabilities']['NEUTRAL'] * 100
    p_pos = res['probabilities']['POSITIVE'] * 100

    icon = '🟢' if label == 'POSITIVE' else ('🔴' if label == 'NEGATIVE' else '🟡')
    print(f"{icon} [{label}] ({conf:.1f}% confidence) | Topic: {topic}")
    print(f"   Câu gốc:   '{text}'")
    print(f"   Xác suất:  [NEG: {p_neg:.1f}% | NEU: {p_neu:.1f}% | POS: {p_pos:.1f}%]\n")


🔍 --- KẾT QUẢ DỰ ĐOÁN TỪNG CÂU VỚI PHOBERT-V2 (Threshold tau=0.08) --- 

🟢 [POSITIVE] (94.4% confidence) | Topic: lecturer
   Câu gốc:   'thầy dạy rất có tâm, giảng bài dễ hiểu và nhiệt tình hỗ trợ sinh viên'
   Xác suất:  [NEG: 4.6% | NEU: 1.1% | POS: 94.4%]

🟢 [POSITIVE] (91.1% confidence) | Topic: program
   Câu gốc:   'môn học cực kỳ bổ ích, em học được rất nhiều kiến thức thực tế'
   Xác suất:  [NEG: 7.9% | NEU: 1.0% | POS: 91.1%]

🔴 [NEGATIVE] (94.8% confidence) | Topic: facility
   Câu gốc:   'máy lạnh phòng wzjwz208 bị hỏng suốt cả học kỳ mà không ai sửa doubledot'
   Xác suất:  [NEG: 94.8% | NEU: 2.0% | POS: 3.2%]

🔴 [NEGATIVE] (89.8% confidence) | Topic: lecturer
   Câu gốc:   'slide bài giảng quá sơ sài, thầy giảng buồn ngủ và khó hiểu fraction không có thực hành'
   Xác suất:  [NEG: 89.8% | NEU: 0.7% | POS: 9.4%]

🟡 [NEUTRAL] (95.6% confidence) | Topic: others
   Câu gốc:   'bài tập trên lớp .'
   Xác suất:  [NEG: 2.1% | NEU: 95.6% | POS: 2.3%]

🟡 [NEUTRAL] (95.7% confidenc

In [8]:
# Cell 5: Batch Inference & Đo Lường Độ Trễ (Latency Benchmark)
import time
import pandas as pd

# Tạo tập mẫu giả lập 100 câu nhận xét
synthetic_comments = [
    ('giáo viên hỗ trợ rất nhiệt tình', 'lecturer'),
    ('phòng học quá nóng và ồn', 'facility'),
    ('thuyết trình nhóm .', 'others'),
    ('nội dung bài giảng ổn áp', 'program'),
    ('không có ý kiến gì thêm', 'others')
] * 20  # 100 câu

sentences_list = [c[0] for c in synthetic_comments]
topics_list = [c[1] for c in synthetic_comments]

print(f'⚡ Đang chạy Batch Inference cho {len(sentences_list)} câu...')
start_time = time.time()
predictions = predictor.predict_batch(sentences_list, topics=topics_list, batch_size=32)
total_time = time.time() - start_time
latency_per_sample = (total_time / len(sentences_list)) * 1000

print(f'⏱️ Tổng thời gian: {total_time:.3f} giây')
print(f'🚀 Tốc độ xử lý:   {len(sentences_list) / total_time:.1f} câu/giây')
print(f'🎯 Độ trễ trung bình: {latency_per_sample:.2f} ms/câu')

# Chuyển kết quả sang DataFrame
df_results = pd.DataFrame(predictions)
df_results.head(10)


⚡ Đang chạy Batch Inference cho 100 câu...
⏱️ Tổng thời gian: 0.819 giây
🚀 Tốc độ xử lý:   122.2 câu/giây
🎯 Độ trễ trung bình: 8.19 ms/câu


,raw_sentence,topic,predicted_label,confidence,prob_negative,prob_neutral,prob_positive
0,giáo viên hỗ trợ rất nhiệt tình,lecturer,POSITIVE,0.944348,0.044495,0.011157,0.944348
1,phòng học quá nóng và ồn,facility,NEGATIVE,0.898375,0.898375,0.008551,0.093074
2,thuyết trình nhóm .,others,NEUTRAL,0.961543,0.015887,0.961543,0.022570
3,nội dung bài giảng ổn áp,program,POSITIVE,0.949845,0.036805,0.013350,0.949845
4,không có ý kiến gì thêm,others,NEUTRAL,0.906656,0.009523,0.906656,0.083820
5,giáo viên hỗ trợ rất nhiệt tình,lecturer,POSITIVE,0.944348,0.044495,0.011157,0.944348
6,phòng học quá nóng và ồn,facility,NEGATIVE,0.898375,0.898375,0.008551,0.093074
7,thuyết trình nhóm .,others,NEUTRAL,0.961543,0.015887,0.961543,0.022570
8,nội dung bài giảng ổn áp,program,POSITIVE,0.949845,0.036805,0.013350,0.949845
9,không có ý kiến gì thêm,others,NEUTRAL,0.906656,0.009523,0.906656,0.083820


In [9]:
# Cell 6: Xuất Kết Quả Dự Đoán Ra File CSV
output_csv = 'phobert_predictions_demo.csv'
df_results.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f'💾 Đã xuất file kết quả: {output_csv} ({len(df_results)} dòng)')


💾 Đã xuất file kết quả: phobert_predictions_demo.csv (100 dòng)
